# SOFR prediction + email (Jupyter / DataZone)

1. `git pull origin main` en el clone
2. Rellena SMTP en la celda **Email** (o en `curve_config.py`)
3. Corre las celdas en orden

**Frecuencia del loop (terminal):** `curve_config.PREDICTION_EVERY_MINUTES` (default 30).
**Lookback gráficos:** `curve_config.GRAPH_LOOKBACK_DAYS` (días de calendario, p.ej. 9).

Outputs: `sofr_live.csv`, `predictions_latest.csv`, `graphs/evolution/*.png`

In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import Image, display
import pandas as pd

logging.disable(logging.WARNING)
logging.getLogger().setLevel(logging.ERROR)
for _n in list(logging.root.manager.loggerDict):
    lg = logging.getLogger(_n)
    if any(k in str(_n).lower() for k in ("bbg", "blp", "dz", "datazone", "activity", "ms")):
        lg.disabled = True
        lg.handlers = []

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "run_prediction.py").exists() else _DZ
if not (REPO / "run_prediction.py").exists():
    raise FileNotFoundError("No encuentro run_prediction.py en %s" % REPO)

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())
print("python:", sys.executable)

In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
# ── Opciones de esta corrida ─────────────────────────────────────────────────
REFRESH = False          # True = refresh BBG; False = usa sofr_live.csv (rápido para probar email)
PULL_HISTORY = False     # True = reconstruye historia bdib (lento)
SEND_EMAIL = True
MAKE_GRAPHS = True

import curve_config
importlib.reload(curve_config)

print("PREDICTION_EVERY_MINUTES:", curve_config.PREDICTION_EVERY_MINUTES)
print("GRAPH_LOOKBACK_DAYS:", curve_config.GRAPH_LOOKBACK_DAYS)
print("EMAIL_ENABLED:", curve_config.EMAIL_ENABLED, "TO:", curve_config.REPORT_TO)

In [ ]:
# ── Email: RELLENA ESTO para la prueba ───────────────────────────────────────
# (override en memoria; no hace falta editar el archivo si solo pruebas)

curve_config.EMAIL_ENABLED = True
curve_config.SMTP_HOST = ""          # <-- p.ej. relay interno MS
curve_config.SMTP_PORT = 587
curve_config.SMTP_USE_TLS = True     # False si el relay no usa TLS
curve_config.SMTP_USER = ""          # vacío si no pide login
curve_config.SMTP_PASSWORD = ""
curve_config.REPORT_FROM = ""        # tu email
curve_config.REPORT_TO = []          # ["tu@firm.com"]

# Sync modules that cached imports at load time
import models
importlib.reload(models)
models.EMAIL_ENABLED = curve_config.EMAIL_ENABLED
models.SMTP_HOST = curve_config.SMTP_HOST
models.SMTP_PORT = curve_config.SMTP_PORT
models.SMTP_USER = curve_config.SMTP_USER
models.SMTP_PASSWORD = curve_config.SMTP_PASSWORD
models.REPORT_FROM = curve_config.REPORT_FROM
models.REPORT_TO = list(curve_config.REPORT_TO)
models.SMTP_USE_TLS = curve_config.SMTP_USE_TLS
models.EMAIL_ATTACH_CSV = True
models.EMAIL_ATTACH_GRAPHS = True

print(
    "email ready?",
    "ENABLED=", curve_config.EMAIL_ENABLED,
    "HOST=", repr(curve_config.SMTP_HOST),
    "FROM=", repr(curve_config.REPORT_FROM or curve_config.SMTP_USER),
    "TO=", curve_config.REPORT_TO,
)
assert curve_config.SMTP_HOST and curve_config.REPORT_TO and (
    curve_config.REPORT_FROM or curve_config.SMTP_USER
), "Rellena SMTP_HOST, REPORT_TO y REPORT_FROM/SMTP_USER arriba"

In [ ]:
# Predicción + graphs + email
import data
import run_prediction

importlib.reload(data)
importlib.reload(run_prediction)

table = run_prediction.run_once(
    refresh=REFRESH,
    force_history=PULL_HISTORY,
    send_email=SEND_EMAIL,
    make_graphs=MAKE_GRAPHS,
)

from models import PREDICTION_COLS

cols = [c for c in PREDICTION_COLS if c in table.columns]
display(table[cols])
print("email body preview:\n", run_prediction.LAST_TEXT[:800])

In [ ]:
# Mostrar PNGs generados
evo = Path(getattr(curve_config, "GRAPH_DIR_EVOLUTION", "graphs/evolution"))
pngs = sorted(evo.glob("*.png")) if evo.exists() else []
print("graphs:", evo.resolve(), "n=", len(pngs))
for p in pngs[:6]:
    print(p.name)
    display(Image(filename=str(p)))

Si ves `[email sent]` arriba, revisa la bandeja. Si `[email skipped]` / `[email FAILED]`, corrige SMTP en la celda Email y vuelve a correr esa celda + la de predicción.

Para cambiar lookback de gráficos sin editar archivo:
```python
curve_config.GRAPH_LOOKBACK_DAYS = 9
```